# Chapter 17 — Bias, Variance, and Regularization

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then the errata page.

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/USER/from-absolute-zero/main/requirements.txt  # Colab only; skip locally

## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
items = [("Cold Brew Concentrate", 12.50, "Beverage"),
         ("Ceramic Mug", 8.75, "Merch"),
         ("Espresso Beans 1kg", 24.00, "Beans"),
         ("Paper Filters x100", 4.25, "Supplies"),
         ("Travel Tumbler", 18.90, "Merch"),
         ("Decaf Beans 1kg", 22.50, "Beans"),
         ("Milk Frother", 31.00, "Equipment"),
         ("Gift Card", 25.00, "Other")]
ctry = ["United Kingdom", "Germany", "France", "Netherlands", "Ireland"]
rows, inv = [], 536000
for month in range(1, 13):
    for _ in range(int(rng.integers(55, 85))):
        inv += 1
        c = str(rng.choice(ctry, p=[.55, .15, .12, .10, .08]))
        cust = int(rng.integers(12000, 12400))
        day = int(rng.integers(1, 29))
        for _ in range(int(rng.integers(1, 4))):
            i = int(rng.integers(0, len(items)))
            rows.append({"InvoiceNo": str(inv), "StockCode": "S%d" % (1000 + i),
                "Description": items[i][0], "Category": items[i][2],
                "Quantity": int(rng.integers(1, 13)),
                "InvoiceDate": "2024-%02d-%02d" % (month, day),
                "UnitPrice": items[i][1], "CustomerID": cust, "Country": c})
df = pd.DataFrame(rows)
cancel = df.sample(18, random_state=3).index
df.loc[cancel, "InvoiceNo"] = "C" + df.loc[cancel, "InvoiceNo"]
df.loc[cancel, "Quantity"] = -df.loc[cancel, "Quantity"]
df.loc[df.sample(40, random_state=5).index, "CustomerID"] = np.nan
df.to_csv("retail.csv", index=False)
print(f"wrote retail.csv: {len(df):,} rows, {df.InvoiceNo.nunique():,} invoices")

## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session.

In [ ]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
rng = np.random.default_rng(0)

## The chapter code

### Block 1  (`c1.py`)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def truth(x):                       # the relationship, unknown to any model
    return np.sin(1.4 * x) + 0.3 * x

def sample(n=40):
    x = rng.uniform(-3, 3, n)
    return x, truth(x) + rng.normal(0, 0.45, n)

xs = np.linspace(-3, 3, 200)        # where we measure
runs = 300                          # 300 alternative worlds
NOISE = 0.45 ** 2

print(f"{'degree':>7}{'bias^2':>9}{'variance':>10}{'noise':>8}{'total':>10}")
for deg in [1, 2, 3, 5, 7, 9, 12]:
    preds = np.zeros((runs, len(xs)))
    for r in range(runs):
        x, y = sample()
        preds[r] = np.polyval(np.polyfit(x, y, deg), xs)
    bias2 = ((preds.mean(0) - truth(xs)) ** 2).mean()
    var = preds.var(0).mean()
    print(f"{deg:>7}{bias2:>9.3f}{var:>10.3f}{NOISE:>8.3f}"
          f"{bias2 + var + NOISE:>10.3f}")

### Block 2  (`c2.py`)

In [ ]:
# The two diagnostics that tell you which problem you have.
X = rng.uniform(-3, 3, size=(30, 1))
y = (np.sin(1.4 * X[:, 0]) + 0.3 * X[:, 0]
     + rng.normal(0, 0.45, 30))
cv = KFold(5, shuffle=True, random_state=0)

print(f"{'degree':>7}{'train MSE':>11}{'val MSE':>10}   diagnosis")
for deg in [1, 3, 5, 9, 14]:
    m = make_pipeline(PolynomialFeatures(deg), StandardScaler(),
                      LinearRegression())
    val = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_squared_error").mean()
    m.fit(X, y)
    tr = ((y - m.predict(X)) ** 2).mean()
    d = ("underfit (high bias)" if tr > 0.35 else
         "overfit (high variance)" if val > 2 * tr + 0.1 else "balanced")
    print(f"{deg:>7}{tr:>11.3f}{val:>10.3f}   {d}")

### Block 3  (`c3.py`)

In [ ]:
# The same overfit model, rescued by a penalty instead of less capacity.
X = rng.uniform(-3, 3, size=(30, 1))
y = np.sin(1.4 * X[:, 0]) + 0.3 * X[:, 0] + rng.normal(0, 0.45, 30)
cv = KFold(5, shuffle=True, random_state=0)

print(f"{'alpha':>9}{'train MSE':>11}{'val MSE':>10}{'largest |w|':>13}")
for a in [0.0, 1e-4, 1e-2, 1e-1, 1.0, 10.0, 100.0]:
    m = make_pipeline(PolynomialFeatures(14), StandardScaler(),
                      Ridge(alpha=a) if a else LinearRegression())
    val = -cross_val_score(m, X, y, cv=cv,
                           scoring="neg_mean_squared_error").mean()
    m.fit(X, y)
    tr = ((y - m.predict(X)) ** 2).mean()
    w = np.abs(m[-1].coef_).max()
    print(f"{a:>9.4f}{tr:>11.3f}{val:>10.3f}{w:>13.1f}")

### Block 4  (`c4.py`)

In [ ]:
import pandas as pd
df = pd.read_csv("retail.csv")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
clean = df[~df["InvoiceNo"].str.startswith("C")].copy()

o = clean.groupby("InvoiceNo").agg(
        rev=("Revenue", "sum"), units=("Quantity", "sum"),
        lines=("StockCode", "count"),
        avg_price=("UnitPrice", "mean")).reset_index()

# Two features that genuinely matter, one near-duplicate of units,
# and six columns of pure noise -- the shape of a real feature table.
o["units_copy"] = o["units"] * 1.02 + rng.normal(0, .3, len(o))
for j in range(6):
    o[f"noise_{j}"] = rng.normal(0, 1, len(o))

feats = ["units", "lines", "avg_price", "units_copy"] + \
        [f"noise_{j}" for j in range(6)]
print(f"{len(o):,} orders, {len(feats)} candidate features")
print(f"correlation units vs units_copy: "
      f"{o['units'].corr(o['units_copy']):.3f}")

### Block 5  (`c5.py`)

In [ ]:
Xo, yo = o[feats].values, o["rev"].values
cv = KFold(5, shuffle=True, random_state=0)

def run(model):
    m = make_pipeline(StandardScaler(), model)
    mse = -cross_val_score(m, Xo, yo, cv=cv,
                           scoring="neg_mean_squared_error").mean()
    m.fit(Xo, yo)
    return mse, m[-1].coef_

print(f"{'model':<18}{'CV MSE':>10}{'non-zero':>10}")
for name, mdl in [("plain", LinearRegression()),
                  ("ridge  (a=10)", Ridge(alpha=10)),
                  ("lasso  (a=1)", Lasso(alpha=1.0, max_iter=50000)),
                  ("lasso  (a=5)", Lasso(alpha=5.0, max_iter=50000))]:
    mse, w = run(mdl)
    print(f"{name:<18}{mse:>10.1f}{int((np.abs(w) > 1e-6).sum()):>10}")

print()
_, w_ridge = run(Ridge(alpha=10))
_, w_lasso = run(Lasso(alpha=5.0, max_iter=50000))
print(f"{'feature':<12}{'ridge':>9}{'lasso':>9}")
for f, a, b in zip(feats, w_ridge, w_lasso):
    print(f"{f:<12}{a:>9.1f}{b:>9.1f}")